In [66]:

from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_ollama import ChatOllama
llm=ChatOllama(model="llama3.1:8b")#ollama pull llama3.1:8b
from langgraph.checkpoint.memory import InMemorySaver


In [67]:
class JOKESTATE(TypedDict):
    topic:str
    joke:str
    explanation:str
    

In [68]:
def generate_joke(state:JOKESTATE)->JOKESTATE:
    prompt=f"generate a joke on the topic{state['topic']}"
    ans=llm.invoke(prompt).content
    return {
        'joke':ans
    }

In [69]:
def Explanation_JOKE(state:JOKESTATE)->JOKESTATE:
    prompt=f"Generate a explantion of the joke in detail and explain the context of this {state['joke']}"
    response=llm.invoke(prompt).content
    return {'explanation':response}


In [70]:
graph=StateGraph(JOKESTATE)
graph.add_node("generate_joke",generate_joke)
graph.add_node("Explanation_JOKE",Explanation_JOKE)
#add edge expects node names not the actual function
graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','Explanation_JOKE')
graph.add_edge('Explanation_JOKE',END)
checkpointer = InMemorySaver()#InMemorysaver stores the data in RAM
workflow = graph.compile(checkpointer=checkpointer)


In [71]:
config={'configurable':{'thread_id':'1'}}# in the case the data will be be stored against 1
workflow.invoke({'topic':'Trump'},config=config)#when we implement persistance  ,while executing we have to pass a thread id against which value/data can be stored

{'topic': 'Trump',
 'joke': "Here's one:\n\nWhy did Donald Trump bring a ladder to the White House?\n\nBecause he wanted to take his ego to new heights! (get it?)",
 'explanation': 'The joke relies on wordplay, using multiple meanings of "heights" to create humor. "New heights" can refer to both achieving greater success or status and physically ascending higher.\n\nIn this context, the joke targets Donald Trump\'s reputation for having an inflated sense of self-importance (ego). By bringing a ladder to the White House, Trump is seen as taking his ego literally, making it a physical object that he\'s elevating. This creates a humorous contrast between Trump\'s ego being a figurative concept and him acting as if it\'s something tangible that needs to be physically elevated.\n\nThe joke relies on audience familiarity with Donald Trump\'s public persona and the common perception of him having an oversized ego. The humor comes from seeing this characteristic taken literally, making it a li

In [73]:
final_state=workflow.get_state(config)

In [74]:
final_state.values

{'topic': 'Trump',
 'joke': "Here's one:\n\nWhy did Donald Trump bring a ladder to the White House?\n\nBecause he wanted to take his ego to new heights! (get it?)",
 'explanation': 'The joke relies on wordplay, using multiple meanings of "heights" to create humor. "New heights" can refer to both achieving greater success or status and physically ascending higher.\n\nIn this context, the joke targets Donald Trump\'s reputation for having an inflated sense of self-importance (ego). By bringing a ladder to the White House, Trump is seen as taking his ego literally, making it a physical object that he\'s elevating. This creates a humorous contrast between Trump\'s ego being a figurative concept and him acting as if it\'s something tangible that needs to be physically elevated.\n\nThe joke relies on audience familiarity with Donald Trump\'s public persona and the common perception of him having an oversized ego. The humor comes from seeing this characteristic taken literally, making it a li

In [78]:
#for intermediate statevalues
list(workflow.get_state_history(config=config))
#intially the state will be empty then  start to genrate joke then genrate joke->explantion and finally explation ->end
 

[StateSnapshot(values={'topic': 'Trump', 'joke': "Here's one:\n\nWhy did Donald Trump bring a ladder to the White House?\n\nBecause he wanted to take his ego to new heights! (get it?)", 'explanation': 'The joke relies on wordplay, using multiple meanings of "heights" to create humor. "New heights" can refer to both achieving greater success or status and physically ascending higher.\n\nIn this context, the joke targets Donald Trump\'s reputation for having an inflated sense of self-importance (ego). By bringing a ladder to the White House, Trump is seen as taking his ego literally, making it a physical object that he\'s elevating. This creates a humorous contrast between Trump\'s ego being a figurative concept and him acting as if it\'s something tangible that needs to be physically elevated.\n\nThe joke relies on audience familiarity with Donald Trump\'s public persona and the common perception of him having an oversized ego. The humor comes from seeing this characteristic taken liter

In [80]:
config2={'configurable':{'thread_id':'2'}}

In [81]:
response=workflow.invoke({'topic':"Narendra MODI"},config=config2)#return state

In [85]:
response['joke']

"Here's one:\n\nWhy did Narendra Modi bring a ladder to the meeting?\n\nBecause he wanted to take his governance to the next level! (get it?)"

In [86]:
response['explanation']

'A classic example of wordplay humor!\n\n**Context:** The joke assumes that the listener is familiar with Narendra Modi, who is the current Prime Minister of India. As a public figure, his policies and governance are often scrutinized by the media and the public.\n\n**Explantation:**\n\nThe joke relies on a play on words, using the phrase "take it to the next level" in two different ways.\n\n* In one sense, "taking it to the next level" is a common idiomatic expression meaning to improve or excel at something. In this context, Narendra Modi\'s action of bringing a ladder could metaphorically mean that he wants to elevate his governance to a higher standard.\n* However, literally, taking a ladder to a meeting is absurd, as ladders are used for physical ascension, not for discussing politics.\n\nThe punchline "because he wanted to take his governance to the next level" is a clever play on words. It takes the idiomatic expression and gives it a literal meaning, referencing the physical ac

In [89]:
final_state=workflow.get_state(config2)

In [90]:
final_state.values

{'topic': 'Narendra MODI',
 'joke': "Here's one:\n\nWhy did Narendra Modi bring a ladder to the meeting?\n\nBecause he wanted to take his governance to the next level! (get it?)",
 'explanation': 'A classic example of wordplay humor!\n\n**Context:** The joke assumes that the listener is familiar with Narendra Modi, who is the current Prime Minister of India. As a public figure, his policies and governance are often scrutinized by the media and the public.\n\n**Explantation:**\n\nThe joke relies on a play on words, using the phrase "take it to the next level" in two different ways.\n\n* In one sense, "taking it to the next level" is a common idiomatic expression meaning to improve or excel at something. In this context, Narendra Modi\'s action of bringing a ladder could metaphorically mean that he wants to elevate his governance to a higher standard.\n* However, literally, taking a ladder to a meeting is absurd, as ladders are used for physical ascension, not for discussing politics.\n\

In [91]:
list(workflow.get_state_history(config=config2))

[StateSnapshot(values={'topic': 'Narendra MODI', 'joke': "Here's one:\n\nWhy did Narendra Modi bring a ladder to the meeting?\n\nBecause he wanted to take his governance to the next level! (get it?)", 'explanation': 'A classic example of wordplay humor!\n\n**Context:** The joke assumes that the listener is familiar with Narendra Modi, who is the current Prime Minister of India. As a public figure, his policies and governance are often scrutinized by the media and the public.\n\n**Explantation:**\n\nThe joke relies on a play on words, using the phrase "take it to the next level" in two different ways.\n\n* In one sense, "taking it to the next level" is a common idiomatic expression meaning to improve or excel at something. In this context, Narendra Modi\'s action of bringing a ladder could metaphorically mean that he wants to elevate his governance to a higher standard.\n* However, literally, taking a ladder to a meeting is absurd, as ladders are used for physical ascension, not for dis

In [92]:
#Time travel

In [94]:
workflow.get_state({'configurable': {'thread_id': '2'},'checkpoint_id':'1f1629ee-691b-6538-8001-f662f609b7d0'})

StateSnapshot(values={'topic': 'Narendra MODI', 'joke': "Here's one:\n\nWhy did Narendra Modi bring a ladder to the meeting?\n\nBecause he wanted to take his governance to the next level! (get it?)", 'explanation': 'A classic example of wordplay humor!\n\n**Context:** The joke assumes that the listener is familiar with Narendra Modi, who is the current Prime Minister of India. As a public figure, his policies and governance are often scrutinized by the media and the public.\n\n**Explantation:**\n\nThe joke relies on a play on words, using the phrase "take it to the next level" in two different ways.\n\n* In one sense, "taking it to the next level" is a common idiomatic expression meaning to improve or excel at something. In this context, Narendra Modi\'s action of bringing a ladder could metaphorically mean that he wants to elevate his governance to a higher standard.\n* However, literally, taking a ladder to a meeting is absurd, as ladders are used for physical ascension, not for disc

In [95]:
#now we have reached the  state where checkpint_id is 1f1629ee-691b-6538-8001-f662f609b7d0 which is the state after genrating joke and before explantion so we can see the joke but not the explantion
#from here we can again move forward adnd get explantion or we can move back and change the joke and then move foreard
workflow.invoke(None,config={'configurable':{'thread_id':'2'},'checkpoint_id': '1f1629ee-691b-6538-8001-f662f609b7d0'})


{'topic': 'Narendra MODI',
 'joke': "Here's one:\n\nWhy did Narendra Modi bring a ladder to the meeting?\n\nBecause he wanted to take his governance to the next level! (get it?)",
 'explanation': 'A clever play on words. Let\'s break down the joke step by step.\n\n**The Setup**: The joke starts with the question "Why did Narendra Modi bring a ladder to the meeting?" This is a classic setup for a joke, creating curiosity and anticipation in the listener.\n\n**The Puns**: The punchline contains two layers of wordplay:\n\n1. **Literal meaning**: A ladder is typically used to reach high places or access something that\'s out of reach.\n2. **Idiomatic expression**: "Take it to the next level" is a common idiomatic phrase used in business and politics, meaning to improve performance, achieve greater success, or elevate one\'s position.\n\n**The Connection**: The joke relies on the listener being familiar with Narendra Modi, who is an Indian politician and the current Prime Minister of India.

In [99]:

workflow.get_state({'configurable': {'thread_id': '2'},'checkpoint_id': '1f1629ed-eb8e-6e44-bfff-2cfb01cb3a4b'})

StateSnapshot(values={'topic': 'Narendra MODI', 'joke': "Here's one:\n\nWhy did Narendra Modi bring a ladder to the meeting?\n\nBecause he wanted to take his governance to the next level! (get it?)", 'explanation': 'A clever play on words. Let\'s break down the joke step by step.\n\n**The Setup**: The joke starts with the question "Why did Narendra Modi bring a ladder to the meeting?" This is a classic setup for a joke, creating curiosity and anticipation in the listener.\n\n**The Puns**: The punchline contains two layers of wordplay:\n\n1. **Literal meaning**: A ladder is typically used to reach high places or access something that\'s out of reach.\n2. **Idiomatic expression**: "Take it to the next level" is a common idiomatic phrase used in business and politics, meaning to improve performance, achieve greater success, or elevate one\'s position.\n\n**The Connection**: The joke relies on the listener being familiar with Narendra Modi, who is an Indian politician and the current Prime

In [101]:
workflow.update_state(
    {
        'configurable': {
            'thread_id': '2',
            'checkpoint_ns': '',
            'checkpoint_id': '1f1629ed-eb8e-6e44-bfff-2cfb01cb3a4b'
        }
    },
    {'topic': 'Amitshah'}
)

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f162a1a-f362-6aa8-8000-2b9946c29afa'}}

In [102]:
workflow.invoke(None,config={'configurable':{'thread_id':'2','checkpoint_id': '1f1629ed-eb8e-6e44-bfff-2cfb01cb3a4b'}})

{'topic': 'Narendra MODI',
 'joke': 'Here\'s one:\n\nWhy did Narendra Modi bring a ladder to the speech?\n\nBecause he wanted to take his "Make in India" slogan to new heights!\n\n(Sorry, I know it\'s a bit of a "Modi-fied" pun)',
 'explanation': 'A play on words! Let\'s break down the joke.\n\n**The setup:** The joke begins by asking "Why did Narendra Modi bring a ladder to the speech?" This question establishes a mundane and ordinary scenario, where we expect a practical reason for bringing a ladder. However, the punchline subverts our expectations.\n\n**The punchline:** The punchline is: "Because he wanted to take his \'Make in India\' slogan to new heights!" Here\'s where the wordplay happens:\n\n* "Make in India" is a slogan coined by Narendra Modi, the Prime Minister of India, as part of his economic development policies. It aims to encourage domestic manufacturing and reduce dependence on foreign goods.\n* The phrase "new heights" has a double meaning here:\n\t+ Literally, if so